# Dummy Agent Library

In this simple example, **we're going to code an Agent from scratch**.

This notebook is part of the <a href="https://www.hf.co/learn/agents-course">Hugging Face Agents Course</a>, a free Course from beginner to expert, where you learn to build Agents.

<img src="https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png" alt="Agent Course"/>

In [8]:
!pip install -q huggingface_hub openai

## Serverless API

In the Hugging Face ecosystem, there is a convenient feature called Serverless API that allows you to easily run inference on many models. There's no installation or deployment required.

To run this notebook, **you need a Hugging Face token** that you can get from https://hf.co/settings/tokens. A "Read" token type is sufficient.
- If you are running this notebook on Google Colab, you can set it up in the "settings" tab under "secrets". Make sure to call it "HF_TOKEN" and restart the session to load the environment variable (Runtime -> Restart session).
- If you are running this notebook locally, you can set it up as an [environment variable](https://huggingface.co/docs/huggingface_hub/en/package_reference/environment_variables). Make sure you restart the kernel after installing or updating huggingface_hub. You can update huggingface_hub by modifying the above `!pip install -q huggingface_hub -U`

In [13]:
import os
from openai import OpenAI
from google.colab import userdata

# Se usa Gemini (gratis) en lugar de la API de Hugging Face, por indicación del profesor
client = OpenAI(
    api_key=userdata.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    timeout=60,
    max_retries=0,
)
MODEL ="gemini-4-hydrogen"

We use the `chat` method since it is a convenient and reliable way to apply chat templates:

In [14]:
output = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "The capital of France is"},
    ],
    stream=False,
    max_tokens=1024,
    reasoning_effort="low",
)
print(output.choices[0].message.content)

Paris.


## Dummy Agent

In the previous sections, we saw that the **core of an agent library is to append information in the system prompt**.

This system prompt is a bit more complex than the one we saw earlier, but it already contains:

1. **Information about the tools**
2. **Cycle instructions** (Thought → Action → Observation)

In [15]:
# This system prompt is a bit more complex and actually contains the function description already appended.
# Here we suppose that the textual description of the tools has already been appended.

SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {{"location": "New York"}}
}}


ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """

We need to append the user instruction after the system prompt. This happens inside the `chat` method. We can see this process below:

In [16]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
]

print(messages)

[{'role': 'system', 'content': 'Answer the following questions as best you can. You have access to the following tools:\n\nget_weather: Get the current weather in a given location\n\nThe way you use the tools is by specifying a json blob.\nSpecifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).\n\nThe only values that should be in the "action" field are:\nget_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}\nexample use :\n\n{{\n  "action": "get_weather",\n  "action_input": {{"location": "New York"}}\n}}\n\n\nALWAYS use the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about one action to take. Only one action at a time in this format:\nAction:\n\n$JSON_BLOB (inside markdown cell)\n\nObservation: the result of the action. This Observation is unique, complete, and the source of truth.\n... (thi

Let's call the `chat` method!

In [18]:
output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=False,
    max_tokens=200,
    reasoning_effort="low",
)
print(output.choices[0].message.content)

Action:
```json
{
  "action": "get_weather",
  "action_input": {
    "location": "London"
  }
}
```


Do you see the issue?

> At this point, the model is hallucinating, because it's producing a fabricated "Observation" -- a response that it generates on its own rather than being the result of an actual function or tool call.
> To prevent this, we stop generating right before "Observation:".
> This allows us to manually run the function (e.g., `get_weather`) and then insert the real output as the Observation.

In [19]:
# The answer was hallucinated by the model. We need to stop to actually execute the function!
output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    max_tokens=150,
    stop=["Observation:"], # Let's stop before any actual function is called
    reasoning_effort="low",
)

print(output.choices[0].message.content)

Action:
```json
{
  "action": "get_weather",
  "action_input": {"location": "London"}
}
```


Much Better!

Let's now create a **dummy get weather function**. In a real situation you could call an API.

In [20]:
# Dummy function
def get_weather(location):
    return f"the weather in {location} is sunny with low temperatures. \n"

get_weather('London')

'the weather in London is sunny with low temperatures. \n'

Let's concatenate the system prompt, the base prompt, the completion until function execution and the result of the function as an Observation and resume generation.

In [22]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
    {"role": "assistant", "content": output.choices[0].message.content + "Observation:\n" + get_weather('London')},
    # Gemini no permite terminar con un turno del asistente, así que se pide continuar
    {"role": "user", "content": "Continue from the Observation above and give the Final Answer."},
]

output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=False,
    max_tokens=200,
    reasoning_effort="low",
)

print(output.choices[0].message.content)

Thought: I now know the final answer
Final Answer: The weather in London is sunny with low temperatures.


We learned how we can create Agents from scratch using Python code, and we **saw just how tedious that process can be**. Fortunately, many Agent libraries simplify this work by handling much of the heavy lifting for you.

Now, we're ready **to create our first real Agent** using the `smolagents` library.

## Lo que entendí de este notebook

En este notebook armé un agente "a mano", sin usar ninguna librería de agentes, para entender qué pasa por dentro cuando un LLM usa herramientas.

**Paso a paso lo que hice:**

1. **Instalación y conexión.** Instalé `openai` porque en vez de la API de Hugging Face (que pide créditos de pago) usé Gemini, que tiene un endpoint compatible con OpenAI. Guardé mi API key en los *Secrets* de Colab para no dejarla escrita en el código.
2. **Prueba del modelo.** Le mandé "The capital of France is" y respondió *Paris*, o sea que la conexión funcionaba.
3. **System prompt.** Aquí está la clave del agente: en el prompt le explico al modelo que tiene una herramienta (`get_weather`), cómo debe pedirla (un JSON con `action` y `action_input`) y que tiene que seguir el ciclo **Thought → Action → Observation** hasta llegar a un **Final Answer**.
4. **Primera llamada.** Le pregunté por el clima en Londres. El modelo no respondió directamente, sino que pidió usar la herramienta con `{"action": "get_weather", "action_input": {"location": "London"}}`.
5. **Uso de `stop`.** Repetí la llamada cortando en `"Observation:"`. Esto sirve para que el modelo no se invente el resultado de la herramienta, porque la observación la tiene que poner mi código, no el modelo.
6. **Función dummy.** Creé `get_weather`, que siempre responde que en la ciudad está soleado y con temperaturas bajas. En un agente real aquí se llamaría a una API del clima.
7. **Cerrar el ciclo.** Le devolví al modelo su propia acción más la *Observation* con el resultado de la función, y con eso respondió: *"Final Answer: The weather in London is sunny with low temperatures."*

**Lo que me quedó claro:** el LLM no ejecuta nada por sí solo, solo *decide* qué herramienta usar y con qué argumentos. Quien ejecuta la herramienta es el código, y después le devuelve el resultado al modelo para que razone con eso. Eso mismo es lo que automatizan librerías como smolagents.

**Cosas que tuve que cambiar y me parecieron interesantes:**

- Con Gemini 4 Hydrogen el modelo **no alucinó** la respuesta en la primera llamada como dice el curso; se detuvo solo después de pedir la herramienta. Igual entiendo que el `stop` es necesario porque no todos los modelos se comportan así.
- Gemini **no acepta que la conversación termine con un mensaje del asistente** (el curso usa ese truco para que el modelo "continúe"), entonces tuve que agregar un mensaje de usuario pidiéndole que siguiera desde la Observation.
- El modelo `gemini-flash-latest` me dio error 503 por demanda alta, así que lo cambié por `gemini-4-hydrogen`.